In [1]:
import litellm
litellm.ssl_verify = False

In [26]:
import os 
import getpass
def set_if_undefined(var:str):
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
set_if_undefined("GROQ_API_KEY")
set_if_undefined("SERPER_API_KEY")

In [3]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/3xGOgzMOv5jhRsA3A8N9fQ/leftover.py"

--2026-02-15 16:56:15--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/3xGOgzMOv5jhRsA3A8N9fQ/leftover.py
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 588 [application/x-python]
Saving to: ‘leftover.py’

leftover.py         100%[===================>]     588  --.-KB/s    in 0s      

2026-02-15 16:56:16 (35.0 MB/s) - ‘leftover.py’ saved [588/588]



In [4]:
import sys
sys.path.append(".")

In [5]:
from leftover import LeftoversCrew

In [6]:
files = os.listdir('.')
print(files)

['Build a Tool Calling Agent.ipynb', 'Build a Smarter Search with LangChain Context Retrieval.ipynb', 'ReAct- Build Reasoning and Acting AI Agents with LangGraph.ipynb', '.DS_Store', 'Dall-e-image generation.ipynb', 'Build LangGraph Design Patterns- Orchestration _ Evaluation.ipynb', 'story.mp3', 'Human_story.mp3', 'Semantic similarity with FAISS.ipynb', 'Similarity search.ipynb', 'CrewAI 101- Building Multi-Agent AI Systems.ipynb', 'Create a Structured Meal - Grocery Planner with CrewAI.ipynb', 'Company-Policies', 'incontext-learning.ipynb', 'Build a Meeting Assistant with Whisper, LangChain, & Gradio.ipynb', 'Building smarter ai agents.ipynb', 'leftover.py', 'Summarize Private Documents Using RAG, LangChain, and LLMs.ipynb', '__pycache__', 'Visualization using langchain.ipynb', 'SImilarity search chromadb-1.ipynb', 'Image captioning.ipynb', 'Reflection agenyt-langgraph.ipynb', 'cat.png', 'Personal story teller.ipynb', 'sample-meeting.wav', 'AI Powered Data Analysis with LCEL.ipynb', 

In [31]:
from pydantic import BaseModel,Field
from typing import List,Dict,Optional
from IPython.display import display, JSON,Markdown
from datetime import datetime

In [32]:
class GroceryItem(BaseModel):
    """Individual grocery item"""
    name: str = Field(description="Name of the grocery item")
    quantity: str = Field(description="Quantity needed (for example, '2 lbs', '1 gallon')")
    estimated_price: str = Field(description="Estimated price (for example, '$3-5')")
    category: str = Field(description="Store section (for example, 'Produce', 'Dairy')")

In [33]:
sample_item = GroceryItem(
    name="Chicken Breast",
    quantity="2 lbs",
    estimated_price="$8-12",
    category="Meat"
)
sample_item

GroceryItem(name='Chicken Breast', quantity='2 lbs', estimated_price='$8-12', category='Meat')

In [11]:
type(sample_item)

__main__.GroceryItem

In [34]:
# Display structured data
print("🛒 Sample Grocery Item Structure:")
display(JSON(sample_item.model_dump()))

🛒 Sample Grocery Item Structure:


<IPython.core.display.JSON object>

In [35]:
class MealPlan(BaseModel):
    """Simple meal plan"""
    meal_name: str = Field(description="Name of the meal")
    difficulty_level: str = Field(description="'Easy', 'Medium', 'Hard'")
    servings: int = Field(description="Number of people it serves")
    researched_ingredients: List[str] = Field(description="Ingredients found through research")

In [36]:
sample_meal = MealPlan(
    meal_name="Chicken Stir Fry",
    difficulty_level="Easy",
    servings=4,
    researched_ingredients=["chicken breast", "broccoli", "bell peppers", "garlic", "soy sauce", "rice"]
)

In [37]:
print("\n🍽️ Sample Meal Plan Structure:")
display(JSON(sample_meal.model_dump()))


🍽️ Sample Meal Plan Structure:


<IPython.core.display.JSON object>

In [38]:
class ShoppingCategory(BaseModel):
    """Store section with items"""
    section_name: str = Field(description="Store section (for example, 'Produce', 'Dairy')")
    items: List[GroceryItem] = Field(description="Items in this section")
    estimated_total: str = Field(description="Estimated cost for this section")

In [39]:
sample_section = ShoppingCategory(
    section_name="Produce",
    items=[
        GroceryItem(name="Bell Peppers", quantity="3 pieces", estimated_price="$3-4", category="Produce"),
        GroceryItem(name="Onions", quantity="2 lbs", estimated_price="$2-3", category="Produce")
    ],
    estimated_total="$5-7"
)

In [40]:
print("\n🏪 Sample Shopping Section:")
display(JSON(sample_section.model_dump()))


🏪 Sample Shopping Section:


<IPython.core.display.JSON object>

In [41]:
class GroceryShoppingPlan(BaseModel):
    """Complete simplified shopping plan"""
    total_budget: str = Field(description="Total planned budget")
    meal_plans: List[MealPlan] = Field(description="Planned meals")
    shopping_sections: List[ShoppingCategory] = Field(description="Organized by store sections")
    shopping_tips: List[str] = Field(description="Money-saving and efficiency tips")

In [42]:
from crewai import LLM,Agent,Task,Crew,Process
from crewai_tools import SerperDevTool
llm = LLM(model = "groq/llama-3.3-70b-versatile" , api_key = os.environ["GROQ_API_KEY"], max_tokens = 2048 , temperature = 0.7)

In [43]:
meal_planner = Agent(
    role = "Meal plannner and recipe researcher",
    goal = "Search for optimal recipe and create detailed meal plans",
    backstory = " A skilled meal planner who researches the best recipes online, considering dietary needs, cooking skill levels, and budget constraints.",
    tools = [SerperDevTool()],
    verbose = False,
    llm = llm
)

In [44]:
meal_planning_task = Task(
    description=(
        "Search for the best '{meal_name}' recipe for {servings} people within a {budget} budget. "
        "Consider dietary restrictions: {dietary_restrictions} and cooking skill level: {cooking_skill}. "
        "Find recipes that match the skill level and provide complete ingredient lists with quantities."
    ),
    expected_output="A detailed meal plan with researched ingredients, quantities, and cooking instructions appropriate for the skill level.",
    agent=meal_planner,
    output_pydantic=MealPlan,
    output_file="meals.json"
)

In [45]:
meal_planner_crew = Crew(
    agents=[meal_planner],
    tasks=[meal_planning_task],
    process=Process.sequential,  # Ensures tasks are executed in order
    verbose=True
)

meal_planner_result = meal_planner_crew.kickoff(
    inputs={
        "meal_name": "Chicken Stir Fry",
        "servings": 4,
        "budget": "$25",                           
        "dietary_restrictions": ["no nuts"],       
        "cooking_skill": "beginner"                
    }
)
print("✅ Single meal planning completed!")
print("📋 Single Meal Results:")
print(meal_planner_result)

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name:                                                                       │
│  crew                                                                        │
│  ID:                                                                         │
│  6337780c-bea9-4666-89ab-252e506c7fa6                                        │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started              

ERROR:crewai_tools.tools.serper_dev_tool.serper_dev_tool:Error making request to Serper API: Invalid leading whitespace, reserved character(s), or return character(s) in header value: ' 7b6466715f15930e4b1a380bed2fe256749a2f8f'



[CrewAIEventsBus] Warning: Event pairing mismatch. 'tool_usage_finished' closed 
'llm_call_started' (expected 'tool_usage_started')
╭───────────────────────────── 🔧 Tool Error (#1) ─────────────────────────────╮
│                                                                              │
│  Tool Failed                                                                 │
│  Tool: search_the_internet_with_serper                                       │
│  Iteration: 1                                                                │
│  Error: Invalid leading whitespace, reserved character(s), or return         │
│  character(s) in header value: ' 7b6466715f15930e4b1a380bed2fe256749a2f8f'   │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────── ✅ Tool Execution Completed (#1) ─

ERROR:crewai_tools.tools.serper_dev_tool.serper_dev_tool:Error making request to Serper API: Invalid leading whitespace, reserved character(s), or return character(s) in header value: ' 7b6466715f15930e4b1a380bed2fe256749a2f8f'



[CrewAIEventsBus] Warning: Event pairing mismatch. 'tool_usage_finished' closed 
'llm_call_started' (expected 'tool_usage_started')
╭───────────────────────────── 🔧 Tool Error (#2) ─────────────────────────────╮
│                                                                              │
│  Tool Failed                                                                 │
│  Tool: search_the_internet_with_serper                                       │
│  Iteration: 2                                                                │
│  Error: Invalid leading whitespace, reserved character(s), or return         │
│  character(s) in header value: ' 7b6466715f15930e4b1a380bed2fe256749a2f8f'   │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────── ✅ Tool Execution Completed (#2) ─